In [5]:
# Code to look at DSAS .geojson files for example

import json
from pathlib import Path
from collections import Counter
import pandas as pd


def summarize_dict(d, max_items=20):
    """
    Print a compact summary of a dict.
    """
    print(f"type: dict with {len(d)} keys")
    keys = list(d.keys())
    print("keys:", keys[:max_items])
    if len(keys) > max_items:
        print(f"... ({len(keys) - max_items} more keys)")


def summarize_list(lst, max_items=5):
    """
    Print a compact summary of a list.
    """
    print(f"type: list with {len(lst)} items")
    if len(lst) == 0:
        return
    print("first item type:", type(lst[0]).__name__)
    for i, item in enumerate(lst[:max_items]):
        print(f"\nitem {i}:")
        if isinstance(item, dict):
            print("  keys:", list(item.keys())[:20])
        else:
            print(" ", repr(item)[:300])


def inspect_geojson_features(features, name="features"):
    """
    Inspect a GeoJSON-style feature list.
    """
    print(f"\n--- GeoJSON feature summary: {name} ---")
    print("number of features:", len(features))

    geom_types = Counter()
    prop_keys = Counter()

    for feat in features:
        if not isinstance(feat, dict):
            continue

        geom = feat.get("geometry", {})
        gtype = geom.get("type", "None") if isinstance(geom, dict) else "None"
        geom_types[gtype] += 1

        props = feat.get("properties", {})
        if isinstance(props, dict):
            prop_keys.update(props.keys())

    print("geometry types:", dict(geom_types))
    print("property keys (most common first):")
    for k, n in prop_keys.most_common(50):
        print(f"  {k}: {n}")

    # print first few example properties
    print("\nexample feature properties:")
    for i, feat in enumerate(features[:3]):
        props = feat.get("properties", {})
        geom = feat.get("geometry", {})
        print(f"\nfeature {i}")
        print("  geometry type:", geom.get("type") if isinstance(geom, dict) else None)
        if isinstance(props, dict):
            for k, v in list(props.items())[:20]:
                print(f"  {k}: {v}")


def inspect_json_file(json_path):
    """
    Inspect an arbitrary JSON file and print useful structure info.
    """
    json_path = Path(json_path)
    print("\n" + "=" * 90)
    print("FILE:", json_path)

    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    print("top-level python type:", type(data).__name__)

    if isinstance(data, dict):
        summarize_dict(data)

        # common GeoJSON pattern
        if data.get("type") == "FeatureCollection" and "features" in data:
            print("\nDetected GeoJSON FeatureCollection")
            print("collection type:", data.get("type"))
            if "name" in data:
                print("name:", data["name"])
            if "crs" in data:
                print("crs field:", data["crs"])
            inspect_geojson_features(data["features"], name="features")

        # dict of arrays / nested dicts
        else:
            print("\nTop-level value types:")
            for k, v in data.items():
                print(f"  {k}: {type(v).__name__}")

            # optional deeper look
            for k, v in data.items():
                print(f"\n--- key: {k} ---")
                if isinstance(v, dict):
                    summarize_dict(v)
                elif isinstance(v, list):
                    summarize_list(v, max_items=3)
                else:
                    print(repr(v)[:500])

    elif isinstance(data, list):
        summarize_list(data)

        # if list of GeoJSON features without FeatureCollection wrapper
        if len(data) > 0 and isinstance(data[0], dict):
            has_geom = all(("geometry" in x or "properties" in x) for x in data[: min(10, len(data))])
            if has_geom:
                inspect_geojson_features(data, name="top-level list")

    else:
        print("Top-level object preview:")
        print(repr(data)[:1000])


# -------------------------------------------------------------------
# EDIT THESE PATHS
# -------------------------------------------------------------------
baseline_json = "F:/crs/proj/2026_shoreline_analysis/FL_baseline.geojson"
shorelines_json = "F:/crs/proj/2026_shoreline_analysis/FL_shoreline.geojson"

inspect_json_file(baseline_json)
inspect_json_file(shorelines_json)


FILE: F:\crs\proj\2026_shoreline_analysis\FL_baseline.geojson
top-level python type: dict
type: dict with 3 keys
keys: ['type', 'crs', 'features']

Detected GeoJSON FeatureCollection
collection type: FeatureCollection
crs field: {'type': 'name', 'properties': {'name': 'EPSG:32617'}}

--- GeoJSON feature summary: features ---
number of features: 1
geometry types: {'LineString': 1}
property keys (most common first):
  FID: 1
  Id: 1
  Attribute: 1
  Shape_Leng: 1

example feature properties:

feature 0
  geometry type: LineString
  FID: 0
  Id: 1
  Attribute: DSAS sample data onshore baseline
  Shape_Leng: 48034.4139561

FILE: F:\crs\proj\2026_shoreline_analysis\FL_shoreline.geojson
top-level python type: dict
type: dict with 3 keys
keys: ['type', 'crs', 'features']

Detected GeoJSON FeatureCollection
collection type: FeatureCollection
crs field: {'type': 'name', 'properties': {'name': 'EPSG:32617'}}

--- GeoJSON feature summary: features ---
number of features: 117
geometry types: {'Li

In [9]:
import json
from pathlib import Path
import pandas as pd


def extract_feature_table(json_path, max_rows=10):
    """
    If JSON is GeoJSON-like, flatten feature properties into a table
    and add geometry type.
    """
    json_path = Path(json_path)
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if isinstance(data, dict) and data.get("type") == "FeatureCollection":
        features = data.get("features", [])
    elif isinstance(data, list):
        features = data
    else:
        print(f"{json_path}: not recognized as feature list / FeatureCollection")
        return None

    rows = []
    for i, feat in enumerate(features):
        row = {"feature_index": i}

        geom = feat.get("geometry", {})
        if isinstance(geom, dict):
            row["geometry_type"] = geom.get("type")
            coords = geom.get("coordinates")
            if isinstance(coords, list):
                row["n_coords_top"] = len(coords)

        props = feat.get("properties", {})
        if isinstance(props, dict):
            row.update(props)

        rows.append(row)

    df = pd.DataFrame(rows)

    print("\n" + "=" * 90)
    print("FILE:", json_path)
    print("shape:", df.shape)
    print("\ncolumns:")
    print(df.columns.tolist())

    print("\ndtypes:")
    print(df.dtypes)

    print("\nfirst rows:")
    print(df.head(max_rows).to_string())

    return df

baseline_json = "F:/crs/proj/2026_shoreline_analysis/FL_baseline.geojson"
shorelines_json = "F:/crs/proj/2026_shoreline_analysis/FL_shoreline.geojson"

df_base = extract_feature_table(baseline_json)
df_shore = extract_feature_table(shorelines_json)


FILE: F:\crs\proj\2026_shoreline_analysis\FL_baseline.geojson
shape: (1, 7)

columns:
['feature_index', 'geometry_type', 'n_coords_top', 'FID', 'Id', 'Attribute', 'Shape_Leng']

dtypes:
feature_index      int64
geometry_type     object
n_coords_top       int64
FID                int64
Id                 int64
Attribute         object
Shape_Leng       float64
dtype: object

first rows:
   feature_index geometry_type  n_coords_top  FID  Id                          Attribute    Shape_Leng
0              0    LineString            84    0   1  DSAS sample data onshore baseline  48034.413956

FILE: F:\crs\proj\2026_shoreline_analysis\FL_shoreline.geojson
shape: (117, 10)

columns:
['feature_index', 'geometry_type', 'n_coords_top', 'FID', 'FID_', 'Date_', 'Uncy', 'Year_', 'Attribute', 'Shape_Leng']

dtypes:
feature_index      int64
geometry_type     object
n_coords_top       int64
FID                int64
FID_               int64
Date_             object
Uncy             float64
Year_      